In [9]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

# Import Required Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
import sentencepiece as spm
from tqdm import tqdm
# from modules.nanogpt import GPT as NanoGPT, GPTConfig as NanoGPTConfig
from nanochat import GPT, GPTConfig
import math
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# Load and Prepare Dataset
# Load tokenizer
sp = spm.SentencePieceProcessor(model_file='../data/tokenizers/fineweb_1024_bpe.model')

# Load a small shard of data (first 100k tokens for experimentation)
def load_data_shard(file_path):
    header_bytes = 256 * np.dtype("<i4").itemsize
    token_bytes = np.dtype("<u2").itemsize
    header = np.fromfile(file_path, dtype="<i4", count=256)
    if header.size != 256 or int(header[0]) != 20240520:
        raise ValueError(f"Unexpected shard header for {file_path}")
    num_tokens = int(header[2])
    tokens_np = np.fromfile(file_path, dtype="<u2", count=min(num_tokens, 100000), offset=header_bytes)  # Limit to 100k
    return torch.from_numpy(tokens_np)

data_file = Path('../data/datasets/fineweb10B_sp1024/fineweb_train_000000.bin')
tokens = load_data_shard(data_file).long()
tokens = tokens[:int(len(tokens) * 0.5)]
print(f"Loaded {len(tokens)} tokens")

# Train-test split
train_size = int(0.8 * len(tokens))
train_tokens = tokens[:train_size]
test_tokens = tokens[train_size:]
print(f"Train tokens: {len(train_tokens)}, Test tokens: {len(test_tokens)}")

Loaded 50000 tokens
Train tokens: 40000, Test tokens: 10000


In [12]:
# Create Dataset and DataLoader
class TokenDataset(Dataset):
    def __init__(self, tokens, seq_len=128):
        self.tokens = tokens
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) - self.seq_len

    def __getitem__(self, idx):
        x = self.tokens[idx:idx + self.seq_len]
        y = self.tokens[idx + 1:idx + self.seq_len + 1]
        return x, y

train_dataset = TokenDataset(train_tokens, seq_len=128)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TokenDataset(test_tokens, seq_len=128)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 39872
Test dataset size: 9872


In [31]:
# Build tokenizer LUTs for BPB calculation
def build_sentencepiece_luts(sp, vocab_size, device):
    table_size = max(sp.vocab_size(), vocab_size)
    base_bytes_np = np.zeros((table_size,), dtype=np.int16)
    has_leading_space_np = np.zeros((table_size,), dtype=np.bool_)
    is_boundary_token_np = np.ones((table_size,), dtype=np.bool_)
    for token_id in range(sp.vocab_size()):
        if sp.is_control(token_id) or sp.is_unknown(token_id) or sp.is_unused(token_id):
            continue
        is_boundary_token_np[token_id] = False
        if sp.is_byte(token_id):
            base_bytes_np[token_id] = 1
            continue
        piece = sp.id_to_piece(token_id)
        if piece.startswith("▁"):
            has_leading_space_np[token_id] = True
            piece = piece[1:]
        base_bytes_np[token_id] = len(piece.encode("utf-8"))
    return (
        torch.tensor(base_bytes_np, dtype=torch.int16, device=device),
        torch.tensor(has_leading_space_np, dtype=torch.bool, device=device),
        torch.tensor(is_boundary_token_np, dtype=torch.bool, device=device),
    )

base_bytes_lut, has_leading_space_lut, is_boundary_token_lut = build_sentencepiece_luts(sp, 1024, device)

# Validation function to compute BPB
def compute_bpb(model, dataloader, base_bytes_lut, has_leading_space_lut, is_boundary_token_lut, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    total_bytes = 0
    with torch.no_grad():
        for x, y in tqdm(dataloader, desc='validation'):
            x, y = x.to(device), y.to(device)
            loss = model(x, targets=y)
            total_loss += loss.item() * y.numel()
            total_tokens += y.numel()
            
            # Calculate bytes
            prev_ids = x.view(-1)
            tgt_ids = y.view(-1)
            token_bytes = base_bytes_lut[tgt_ids].to(dtype=torch.int16)
            token_bytes += (has_leading_space_lut[tgt_ids] & ~is_boundary_token_lut[prev_ids]).to(dtype=torch.int16)
            total_bytes += token_bytes.sum().item()
    
    val_loss = total_loss / total_tokens
    bits_per_token = val_loss / math.log(2.0)
    tokens_per_byte = total_tokens / total_bytes
    bpb = bits_per_token * tokens_per_byte
    model.train()
    return val_loss, bpb

In [32]:
config = GPTConfig(  
    sequence_len=128,
    vocab_size=1024,
    n_layer=6,
    n_head=4,  
    n_kv_head=4,  
    n_embd=256,  
)  

model = GPT(config)  

In [33]:
# Set Up Optimizer and Loss Function
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Training Loop
model.to(device)

epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    i = 0
    for x, y in tqdm(train_dataloader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = model(x, targets=y)
        # loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
        i += 1
        if i % 100 == 0:
            val_loss, val_bpb = compute_bpb(model, test_dataloader, base_bytes_lut, has_leading_space_lut, is_boundary_token_lut, device)
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {(total_loss / i):.4f}, Val Loss: {val_loss:.4f}, Val BPB: {val_bpb:.4f}")
            
    
    train_loss = total_loss / len(train_dataloader)
    val_loss, val_bpb = compute_bpb(model, test_dataloader, base_bytes_lut, has_leading_space_lut, is_boundary_token_lut, device)
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val BPB: {val_bpb:.4f}")

  2%|▏         | 20/1246 [00:45<45:18,  2.22s/it]